In [1]:
# Installing autogluo using this set https://auto.gluon.ai/stable/install.html 
# using the pip option with gpu 
# !pip install autogluon

In [2]:
import sys
import os
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularDataset, TabularPredictor

In [3]:
output_path= '/home/takataka_ju/2024-ESL-BiomassSAR/fdl-2024/output/lband/'
df = pd.read_csv('ml_ready_data_lidar_pcube_lband.csv')

In [4]:
# Filters the DataFrame to include only rows where polarimetry is 0.
# Splits the filtered DataFrame into training and testing datasets.
polarimetry = 0 
work_df= df[df['polarimetry']==polarimetry].copy()
train_data, test_data = train_test_split(work_df, test_size=0.2, random_state=42)

In [5]:
work_df.head()

,x,y,z_0,z_1,z_2,z_3,z_4,z_5,z_6,z_7,...,z_30,z_31,z_32,z_33,z_34,z_35,heading,polarimetry,acquisition_mode,lidar
0,0,260,-328.566147,-159.904252,-158.784616,-14.008676,-12.597413,-8.856641,-9.092836,-8.845550,...,-19.489725,-18.591636,-19.072832,-19.966877,-21.000055,-21.703180,0,0,0,0.0
1,0,261,-68.883497,-65.348985,-10.365919,-13.052322,-12.092961,-11.475720,-11.564435,-10.631142,...,-12.771584,-13.369537,-15.613647,-19.611151,-21.750447,-20.947686,0,0,0,0.0
2,0,262,-14.230014,-7.660654,-8.249750,-10.906433,-14.125485,-16.024520,-16.627897,-15.649481,...,-10.387297,-11.057387,-11.253114,-11.834310,-13.167007,-14.233860,0,0,0,0.0
3,0,263,-2.046507,2.954239,4.361038,4.570612,2.825716,-0.100421,-4.886166,-10.695657,...,-8.389768,-8.960672,-8.017043,-7.142231,-7.485433,-8.124657,0,0,0,0.0
4,0,264,-5.001877,-3.293858,-0.318383,2.342329,3.263034,2.900796,1.448686,-0.741961,...,-9.040266,-9.854778,-8.255973,-6.927560,-6.878281,-7.788928,0,0,0,0.0


In [6]:
train_data.shape, test_data.shape

((312184, 42), (78046, 42))

In [7]:
train_data.columns

Index(['x', 'y', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8',
       'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15', 'z_16', 'z_17',
       'z_18', 'z_19', 'z_20', 'z_21', 'z_22', 'z_23', 'z_24', 'z_25', 'z_26',
       'z_27', 'z_28', 'z_29', 'z_30', 'z_31', 'z_32', 'z_33', 'z_34', 'z_35',
       'heading', 'polarimetry', 'acquisition_mode', 'lidar'],
      dtype='object')

In [8]:
# Sets the proportion of the training data to use for training.
# If train_subset_prop is not 1, it creates a subset of the training data.
train_subset_prop = 0.1

if train_subset_prop != 1:
    train_subset, _ = train_test_split(train_data, test_size=1-train_subset_prop, random_state=42)

In [9]:
# Removes the heading column.
# Filters out rows where the specified label column has missing values.
def transform(df, label ):
    cols = [x for x in df.columns.tolist() if x !='heading'] 
    working_cols = cols
    df = df[working_cols]
    df = df[~df[label].isna()]
    return df

def save_results(results, output_path):
    res_string = str(results)
    # summary = predictor.fit_summary()
    with open(os.path.join(output_path, 'model_summary.txt'), 'w') as f:
        f.write(res_string)

In [15]:
# Transforms the train_subset DataFrame.
train_subset= transform(train_subset, 'lidar')
train_subset 

size_train = len(train_subset) 

In [16]:
train_subset.columns

Index(['x', 'y', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8',
       'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15', 'z_16', 'z_17',
       'z_18', 'z_19', 'z_20', 'z_21', 'z_22', 'z_23', 'z_24', 'z_25', 'z_26',
       'z_27', 'z_28', 'z_29', 'z_30', 'z_31', 'z_32', 'z_33', 'z_34', 'z_35',
       'polarimetry', 'acquisition_mode', 'lidar'],
      dtype='object')

In [23]:
time_limit= 1000
model_preset= 'medium_quality'
label = 'lidar'
excl_models=[]
dataset_name = 'ml_ready_data_lidar_pcube_lband'


output_directory = f"{output_path}/{dataset_name}__{label}__{time_limit}__{model_preset}___tsp_{train_subset_prop}__polarim_{polarimetry}"



predictor = TabularPredictor(label, path=output_directory).fit(train_subset, 
                                                            time_limit=time_limit,
                                                            presets=model_preset,
                                                            excluded_model_types=excl_models
                                                            )

test_data = transform(test_data, label)

# test_data.to_csv(os.path.join(output_directory, 'test_data.csv'), index=False)
y_pred = predictor.predict(test_data.drop(columns=[label]))
results = predictor.evaluate_predictions(y_true=test_data[label], y_pred=y_pred, auxiliary_metrics=True)
size_test = len(test_data)



print(results)

sizett = {'len_train' :size_train, 'len_test':size_test  }
results.update(sizett)

save_results(results, output_directory)
res = predictor.leaderboard(test_data)
res.to_csv(os.path.join(output_directory, 'leaderboard_results.csv'))

pred = predictor.feature_importance(test_data)
pred.to_csv(os.path.join(output_directory, 'feature_importance.csv'))



Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #25~22.04.1-Ubuntu SMP Thu Jun 13 19:41:39 UTC 2024
CPU Count:          8
Memory Avail:       53.81 GB / 62.79 GB (85.7%)
Disk Space Avail:   2691.55 GB / 2907.34 GB (92.6%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1000s
AutoGluon will save models to "/home/takataka_ju/2024-ESL-BiomassSAR/myenv/output/lband//ml_ready_data_lidar_pcube_lband__lidar__1000__medium_quality___tsp_0.1__polarim_0"
Train Data Rows:    31218
Train Data Columns: 40
Label Column:       lidar
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (41.84088947405134, 0.0, 16.09241, 9.38734)
	If 'regression' is not the correct problem_type, please m

[1000]	valid_set's rmse: 3.83974
[2000]	valid_set's rmse: 3.77812
[3000]	valid_set's rmse: 3.76172
[4000]	valid_set's rmse: 3.74826
[5000]	valid_set's rmse: 3.74777


	-3.7461	 = Validation score   (-root_mean_squared_error)
	19.92s	 = Training   runtime
	0.39s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 879.12s of the 879.12s of remaining time.


[1000]	valid_set's rmse: 3.61936
[2000]	valid_set's rmse: 3.5987


	-3.5934	 = Validation score   (-root_mean_squared_error)
	12.93s	 = Training   runtime
	0.19s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 865.83s of the 865.83s of remaining time.
	-4.0456	 = Validation score   (-root_mean_squared_error)
	164.96s	 = Training   runtime
	0.24s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 697.36s of the 697.36s of remaining time.
		Import catboost failed. Numpy version may be outdated, Please ensure numpy version >=1.17.0. If it is not, please try 'pip uninstall numpy -y; pip install numpy>=1.17.0' Detailed info: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
Fitting model: ExtraTreesMSE ... Training model for up to 697.06s of the 697.05s of remaining time.
	-3.9986	 = Validation score   (-root_mean_squared_error)
	21.44s	 = Training   runtime
	0.21s	 = Validation runtime
Fitting model: NeuralNetFastAI ... Training model for

...

	-4.2324	 = Validation score   (-root_mean_squared_error)
	228.21s	 = Training   runtime
	0.15s	 = Validation runtime
Fitting model: LightGBMLarge ... Training model for up to 386.9s of the 386.9s of remaining time.


[1000]	valid_set's rmse: 3.59369
[2000]	valid_set's rmse: 3.58133
[3000]	valid_set's rmse: 3.57884
[4000]	valid_set's rmse: 3.57817
[5000]	valid_set's rmse: 3.57711
[6000]	valid_set's rmse: 3.57725


	-3.577	 = Validation score   (-root_mean_squared_error)
	106.13s	 = Training   runtime
	0.82s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.0s of the 278.04s of remaining time.
	Ensemble Weights: {'KNeighborsDist': 0.4, 'LightGBM': 0.3, 'LightGBMLarge': 0.3}
	-3.3714	 = Validation score   (-root_mean_squared_error)
	0.03s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 722.06s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 52.0 rows/s (2500 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/takataka_ju/2024-ESL-BiomassSAR/myenv/output/lband//ml_ready_data_lidar_pcube_lband__lidar__1000__medium_quality___tsp_0.1__polarim_0")


{'root_mean_squared_error': np.float64(-3.31700101607419), 'mean_squared_error': np.float64(-11.002495740637208), 'mean_absolute_error': np.float64(-2.130452172705382), 'r2': 0.8760260065883045, 'pearsonr': 0.9363876965080692, 'median_absolute_error': np.float64(-1.2829693232945019)}


These features in provided data are not utilized by the predictor and will be ignored: ['polarimetry']
Computing feature importance via permutation shuffling for 39 features using 5000 rows with 5 shuffle sets...
	13665.14s	= Expected runtime (2733.03s per shuffle set)
	17371.91s	= Actual runtime (Completed 5 of 5 shuffle sets)
